# Wave 1 — Module 1B: Peer-Group Wage Benchmarking
**Deteksi indikasi "lapor gaji lebih rendah dari aslinya" dengan membandingkan upah per karyawan ke perusahaan sejenis (cohort).**

Requirement yang dicakup (`requirements.md` #2, #4, #5, #6):
- Upah dilaporkan **jauh di bawah** cohort sejenis (sektor, wilayah, skala) → *perlu dicek*
- **Directional**: upah tinggi tidak pernah di-flag
- **UMR hanya pre-filter**, bukan vonis. Upah di bawah UMR saja **tidak** membuat perusahaan di-flag (itu urusan Disnaker). Flag hanya muncul kalau upahnya juga jauh di bawah **peer-group**-nya.
- Output diurutkan dari paling berisiko + kolom `reason`
- Hanya data level perusahaan; kolom data pribadi ditolak

### File yang dibutuhkan
| File | Wajib? | Dipakai untuk |
|---|---|---|
| `payroll_timeseries.csv` | ✅ | Upah dilaporkan per employer per bulan |
| `employer_master.csv` | ✅ | Sektor, wilayah, skala (+ UMR kalau ada) → cohort key |
| `headcount_timeseries.csv` | opsional | Cadangan kalau payroll tidak punya jumlah karyawan / master tidak punya skala |
| `ground_truth.csv` | opsional | Evaluasi recall & false positive |

`remittance_timeseries.csv` tidak dipakai di sini (itu untuk Module C). Benford's Law layer **sengaja di-skip** (dipindah ke Wave 5).

### Cara pakai
1. **Runtime → Run all**. Izinkan akses Google Drive saat diminta.
2. Cek output sel **3. Deteksi kolom**. Kalau ada yang salah tebak, isi manual `COLUMN_MAP` lalu jalankan ulang dari sel itu.
3. Cek **`POSITIVE_LABELS`** di sel evaluasi (label ground truth untuk kasus under-reporting upah).
4. Lihat tabel sweep, update `ConfigB`, Run all lagi. Output tersimpan ke Drive folder `wave 0.B`.


In [ ]:
import os, glob, re
from dataclasses import dataclass, asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 220)

try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Running di Colab:", IN_COLAB)

## 1. Load data dari Google Drive

In [ ]:
USE_DRIVE = True
BASE_DIR  = "/content/drive/MyDrive/Colab Notebooks/Healthkathon Engine"
DRIVE_DIR = f"{BASE_DIR}/Dummy Healthkathon"   # folder dataset
OUT_DRIVE = f"{BASE_DIR}/wave 0.B"             # folder output Module B

if os.environ.get("HK_DATA_DIR"):              # (untuk testing lokal)
    DATA_DIR = Path(os.environ["HK_DATA_DIR"])
elif IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path(DRIVE_DIR)
    if not (DATA_DIR / "payroll_timeseries.csv").exists():
        hits = glob.glob("/content/drive/MyDrive/**/payroll_timeseries.csv", recursive=True)
        if hits:
            DATA_DIR = Path(hits[0]).parent
            print("⚠ DRIVE_DIR tidak ditemukan, pakai folder yang ketemu:", DATA_DIR)
else:
    DATA_DIR = Path("/content/data")

print("DATA_DIR =", DATA_DIR)
for f in sorted(DATA_DIR.glob("*.csv")):
    print("  ✓", f.name)
for req in ["payroll_timeseries.csv", "employer_master.csv"]:
    assert (DATA_DIR / req).exists(), f"{req} belum ada di DATA_DIR!"


## 2. Intip skema file yang dipakai

In [ ]:
FORBIDDEN_COLS = {"nama", "name", "nama_karyawan", "employee_name", "nik", "no_kartu",
                  "no_ktp", "ktp", "alamat", "address", "tanggal_lahir", "birth_date"}

def check_privacy(df, name):
    bad = FORBIDDEN_COLS & {c.lower() for c in df.columns}
    if bad:
        raise ValueError(f"{name} berisi kolom data pribadi {bad}. Hapus dulu sebelum dipakai.")

RAW = {}
for name in ["payroll_timeseries", "employer_master", "headcount_timeseries", "ground_truth"]:
    p = DATA_DIR / f"{name}.csv"
    if not p.exists():
        print(f"(—) {name}.csv tidak ada, dilewati\n"); continue
    df = pd.read_csv(p); check_privacy(df, name); RAW[name] = df
    print(f"=== {name}.csv  ({len(df):,} baris) ===")
    print(df.dtypes.to_string())
    display(df.head(3)) if "display" in globals() else print(df.head(3))
    print()

## 3. Deteksi kolom (column mapping)
Biarkan `None` untuk auto-detect, atau isi nama kolom manual kalau tebakannya salah.

**Upah**: notebook butuh *upah rata-rata per karyawan*. Urutan prioritas:
1. kolom rata-rata (`pr_avg_wage`), atau
2. kolom total upah (`pr_total_wage`) ÷ jumlah karyawan, atau
3. kalau 1 baris = 1 karyawan (`pr_single_wage`), dirata-rata per employer per bulan.

**UMR**: dicari di payroll (per bulan) → employer_master → `UMR_TABLE` manual. Kalau tidak ada sama sekali, pre-filter UMR otomatis nonaktif (semua employer jadi kandidat).

In [ ]:
COLUMN_MAP = {
    # payroll_timeseries
    "pr_employer": None, "pr_period": None,
    "pr_avg_wage": None, "pr_total_wage": None, "pr_single_wage": None,
    "pr_headcount": None, "pr_umr": None,
    # employer_master
    "em_employer": None, "em_sector": None, "em_region": None, "em_scale": None, "em_umr": None,
    # headcount_timeseries
    "hc_employer": None, "hc_period": None, "hc_count": None,
    # ground_truth
    "gt_employer": None, "gt_label": None,
}

# Kalau UMR tidak ada di data sama sekali, bisa diisi manual per wilayah (nilai Rp/bulan), mis:
# UMR_TABLE = {"Jakarta": 5_396_761, "Bandung": 4_482_914}
UMR_TABLE = {}

CANDIDATES = {
    "employer": ["employer_id", "id_employer", "company_id", "perusahaan_id", "id_perusahaan",
                 "badan_usaha_id", "kode_badan_usaha", "kode_bu", "npp", "employer"],
    "period":   ["period", "periode", "month", "bulan", "year_month", "yearmonth", "date", "tanggal"],
    "avg_wage": ["avg_wage", "average_wage", "mean_wage", "avg_reported_wage", "reported_avg_wage",
                 "avg_salary", "rata_rata_upah", "upah_rata_rata", "wage_per_employee", "avg_upah", "avg_gaji"],
    "total_wage": ["total_wage", "reported_wage_total", "total_reported_wage", "reported_total_wage",
                   "total_upah", "total_gaji", "payroll_total", "total_payroll", "wage_total",
                   "total_salary", "wage_bill", "total_upah_dilaporkan"],
    "single_wage": ["reported_wage", "wage", "upah", "gaji", "salary", "upah_dilaporkan"],
    "headcount": ["headcount", "active_headcount", "n_active", "n_employees", "employee_count",
                  "jumlah_karyawan", "jumlah_peserta", "registered_headcount", "n_peserta"],
    "sector": ["sektor_usaha", "sektor", "sector", "industry", "industri", "business_sector",
               "jenis_usaha", "bidang_usaha", "kbli"],
    "region": ["wilayah", "region", "kab_kota", "kabupaten_kota", "kabupaten", "kota", "city",
               "provinsi", "province", "daerah", "area"],
    "scale": ["skala", "skala_usaha", "size_class", "company_size", "size_category", "ukuran",
              "scale", "size"],
    "umr": ["umr", "ump", "umk", "min_wage", "minimum_wage", "upah_minimum", "regional_min_wage",
            "umr_wilayah"],
    "gt_label": ["fraud_type", "label", "anomaly_type", "injected_fraud", "fraud_label", "scenario",
                 "case_type", "jenis_fraud", "tipe", "is_fraud", "fraud"],
}

def _pick(df, key, override=None, required=True, exclude=()):
    if df is None:
        return None
    if override:
        if override not in df.columns:
            raise KeyError(f"Kolom '{override}' tidak ada. Kolom tersedia: {list(df.columns)}")
        return override
    lower = {c.lower(): c for c in df.columns if c not in exclude}
    for cand in CANDIDATES[key]:
        if cand in lower:
            return lower[cand]
    for cand in CANDIDATES[key]:
        if len(cand) >= 4:
            for lc, orig in lower.items():
                if cand in lc:
                    return orig
    if required:
        raise KeyError(f"Tidak bisa menebak kolom '{key}'. Isi manual di COLUMN_MAP. Kolom: {list(df.columns)}")
    return None

PR, EM = RAW["payroll_timeseries"], RAW["employer_master"]
HCR, GT = RAW.get("headcount_timeseries"), RAW.get("ground_truth")
M = COLUMN_MAP
COLS = {}
COLS["pr_employer"] = _pick(PR, "employer", M["pr_employer"])
COLS["pr_period"]   = _pick(PR, "period", M["pr_period"])
COLS["pr_umr"]      = _pick(PR, "umr", M["pr_umr"], required=False)
used = {COLS["pr_employer"], COLS["pr_period"], COLS["pr_umr"]}
COLS["pr_avg_wage"] = _pick(PR, "avg_wage", M["pr_avg_wage"], required=False, exclude=used)
COLS["pr_total_wage"] = None if COLS["pr_avg_wage"] else _pick(PR, "total_wage", M["pr_total_wage"], required=False, exclude=used)
COLS["pr_single_wage"] = None if (COLS["pr_avg_wage"] or COLS["pr_total_wage"]) else \
    _pick(PR, "single_wage", M["pr_single_wage"], required=False, exclude=used)
used |= {COLS["pr_avg_wage"], COLS["pr_total_wage"], COLS["pr_single_wage"]}
COLS["pr_headcount"] = _pick(PR, "headcount", M["pr_headcount"], required=False, exclude=used)
assert any([COLS["pr_avg_wage"], COLS["pr_total_wage"], COLS["pr_single_wage"]]), \
    "Kolom upah di payroll tidak ketemu — isi manual pr_avg_wage / pr_total_wage / pr_single_wage"

COLS["em_employer"] = _pick(EM, "employer", M["em_employer"])
COLS["em_sector"]   = _pick(EM, "sector", M["em_sector"])
COLS["em_region"]   = _pick(EM, "region", M["em_region"])
COLS["em_umr"]      = _pick(EM, "umr", M["em_umr"], required=False)
COLS["em_scale"]    = _pick(EM, "scale", M["em_scale"], required=False,
                            exclude={COLS["em_employer"], COLS["em_sector"], COLS["em_region"], COLS["em_umr"]})
if HCR is not None:
    COLS["hc_employer"] = _pick(HCR, "employer", M["hc_employer"])
    COLS["hc_period"]   = _pick(HCR, "period", M["hc_period"])
    COLS["hc_count"]    = _pick(HCR, "headcount", M["hc_count"])
if GT is not None:
    COLS["gt_employer"] = _pick(GT, "employer", M["gt_employer"])
    COLS["gt_label"]    = _pick(GT, "gt_label", M["gt_label"])

print("Mapping yang dipakai:")
for k, v in COLS.items():
    print(f"  {k:15s} -> {v}")

## 4. Konfigurasi (bisa di-tuning)
Posisi relatif dihitung di **skala log upah**, dalam satuan **IQR cohort**:
`z = (log(upah) − median_cohort) / IQR_cohort`. Contoh: `z = −1.5` artinya 1.5 IQR di bawah median sejenisnya.

In [ ]:
@dataclass(frozen=True)
class ConfigB:
    Z_THRESHOLD: float = 1.5          # flag kalau posisi median < -Z_THRESHOLD (dalam IQR)
    MIN_PERSIST: float = 0.5          # minimal 50% bulan valid juga di bawah threshold (konsisten, bukan sekali)
    MIN_COHORT_SIZE: int = 5          # cohort < 5 employer -> relaksasi dimensi
    MIN_PERIODS: int = 3              # < 3 bulan data upah -> INSUFFICIENT_DATA
    UMR_PREFILTER_MULT: float | None = 1.5   # kandidat = upah median <= 1.5 x UMR. None = pre-filter mati
    IQR_FLOOR: float = 0.05           # IQR log minimum (~5%) -> cegah z meledak kalau cohort seragam
    SCORE_CAP_MULT: float = 3.0       # score_b_norm = min(score_b / (Z_THRESHOLD*mult), 1)

CFG = ConfigB()

# Urutan relaksasi cohort: kalau cohort terlalu kecil, turun ke level berikutnya
COHORT_LEVELS = [
    ("L0", ["sektor", "wilayah", "skala"]),
    ("L1", ["sektor", "wilayah"]),        # relaksasi skala
    ("L2", ["sektor", "skala"]),          # relaksasi wilayah
    ("L3", ["sektor"]),
    ("L4", []),                           # fallback: semua employer
]
LEVEL_NOTE = {"L0": "", "L1": " (skala direlaksasi)", "L2": " (wilayah direlaksasi)",
              "L3": " (hanya sektor)", "L4": " (pembanding semua employer)"}
asdict(CFG)

## 5. Bangun tabel upah per employer per periode (join payroll + employer_master)

In [ ]:
def to_month(s):
    s2 = s.astype(str).str.strip()
    s2 = s2.where(~s2.str.fullmatch(r"\d{6}"), s2.str[:4] + "-" + s2.str[4:])
    return pd.to_datetime(s2, errors="coerce").dt.to_period("M")

num = lambda s: pd.to_numeric(s, errors="coerce")

def headcount_panel():
    if HCR is None:
        return None
    h = pd.DataFrame({"employer_id": HCR[COLS["hc_employer"]].astype(str),
                      "period": to_month(HCR[COLS["hc_period"]]),
                      "hc_ts": num(HCR[COLS["hc_count"]])})
    return h.groupby(["employer_id", "period"], as_index=False)["hc_ts"].sum()

def build_wage_panel():
    df = pd.DataFrame({"employer_id": PR[COLS["pr_employer"]].astype(str),
                       "period": to_month(PR[COLS["pr_period"]])})
    hc = num(PR[COLS["pr_headcount"]]) if COLS["pr_headcount"] else None
    df["total"], df["avg_w"], df["weight"] = np.nan, np.nan, np.nan
    if COLS["pr_avg_wage"]:                      # sudah rata-rata per karyawan
        df["avg_w"] = num(PR[COLS["pr_avg_wage"]])
        df["weight"] = hc if hc is not None else 1.0
        df["hc"] = hc if hc is not None else np.nan
    elif COLS["pr_total_wage"]:                  # total upah / jumlah karyawan
        df["total"] = num(PR[COLS["pr_total_wage"]])
        df["hc"] = hc if hc is not None else np.nan
    else:                                        # 1 baris = 1 karyawan
        df["avg_w"] = num(PR[COLS["pr_single_wage"]]); df["weight"] = 1.0; df["hc"] = 1.0
    df["umr_p"] = num(PR[COLS["pr_umr"]]) if COLS["pr_umr"] else np.nan
    df["wx"] = df["avg_w"] * df["weight"]
    df["wt"] = df["weight"].where(df["avg_w"].notna())
    df = df.dropna(subset=["period"])
    df = df[df["total"].notna() | df["avg_w"].notna()]

    s_ = lambda x: x.sum(min_count=1)
    agg = df.groupby(["employer_id", "period"], as_index=False).agg(
        total=("total", s_), hc=("hc", s_), wx=("wx", s_), wt=("wt", s_), umr_p=("umr_p", "median"))

    if agg["hc"].isna().any():                   # lengkapi jumlah karyawan dari headcount_timeseries
        h = headcount_panel()
        if h is not None:
            agg = agg.merge(h, on=["employer_id", "period"], how="left")
            agg["hc"] = agg["hc"].fillna(agg["hc_ts"]); agg = agg.drop(columns="hc_ts")
        elif agg["total"].notna().any():
            raise ValueError("Payroll hanya punya total upah tanpa jumlah karyawan, dan headcount_timeseries tidak ada.")
    agg["avg_wage"] = np.where(agg["total"].notna(), agg["total"] / agg["hc"], agg["wx"] / agg["wt"])
    agg = agg[agg["avg_wage"] > 0].drop(columns=["wx", "wt"]).copy()
    agg["log_wage"] = np.log(agg["avg_wage"])
    return agg

def scale_from_headcount(n):
    # kategori BPS berdasarkan jumlah tenaga kerja
    out = pd.cut(n, [0, 4, 19, 99, np.inf], labels=["mikro", "kecil", "menengah", "besar"]).astype(str)
    return out.replace("nan", "tidak_diketahui")

def build_employer_table(panel):
    e = pd.DataFrame({"employer_id": EM[COLS["em_employer"]].astype(str),
                      "sektor": EM[COLS["em_sector"]].astype(str).str.strip(),
                      "wilayah": EM[COLS["em_region"]].astype(str).str.strip()})
    med_hc = panel.groupby("employer_id")["hc"].median()
    if COLS["em_scale"]:
        e["skala"] = EM[COLS["em_scale"]].astype(str).str.strip()
    else:
        e["skala"] = scale_from_headcount(e["employer_id"].map(med_hc))
        print("ℹ employer_master tidak punya kolom skala -> diturunkan dari median jumlah karyawan (kategori BPS)")
    e["umr_m"] = num(EM[COLS["em_umr"]]) if COLS["em_umr"] else np.nan
    if UMR_TABLE:
        e["umr_m"] = e["umr_m"].fillna(e["wilayah"].map(UMR_TABLE))
    e = e.drop_duplicates("employer_id", keep="last")
    return e[e["employer_id"].isin(panel["employer_id"])].reset_index(drop=True)

PANEL = build_wage_panel()
EMP = build_employer_table(PANEL)
PANEL = PANEL.merge(EMP, on="employer_id", how="inner")
PANEL["umr"] = PANEL["umr_p"].fillna(PANEL["umr_m"])
HAS_UMR = PANEL["umr"].notna().any()

print(f"Panel upah: {PANEL.employer_id.nunique():,} employer, {PANEL.period.nunique()} periode "
      f"({PANEL.period.min()} s/d {PANEL.period.max()})")
print(f"Upah rata-rata/karyawan: median Rp{PANEL.avg_wage.median():,.0f}")
if HAS_UMR:
    ratio = (PANEL.avg_wage / PANEL.umr).median()
    print(f"Rasio upah/UMR (median): {ratio:.2f}")
    if ratio > 20:
        print("⚠ Rasio sangat besar — kemungkinan kolom upah yang terpilih adalah TOTAL, bukan per karyawan. Cek COLUMN_MAP.")
else:
    print("⚠ UMR tidak ditemukan -> pre-filter UMR dinonaktifkan (semua employer jadi kandidat)")
missing = set(PR[COLS["pr_employer"]].astype(str)) - set(EMP.employer_id)
if missing:
    print(f"⚠ {len(missing)} employer di payroll tidak ada di employer_master -> dikeluarkan")
PANEL.head()

## 6. Cohort key + relaksasi
Cohort = (sektor, wilayah, skala). Kalau isi cohort < `MIN_COHORT_SIZE` employer, dimensi direlaksasi bertahap: tanpa skala → tanpa wilayah → hanya sektor → semua employer.

In [ ]:
def assign_cohorts(emp, cfg=CFG):
    e = emp.copy()
    e["cohort_level"] = None; e["cohort_size"] = np.nan
    for lvl, keys in COHORT_LEVELS:
        size = e.groupby(keys)["employer_id"].transform("nunique") if keys else pd.Series(len(e), index=e.index)
        m = e["cohort_level"].isna() & (size >= cfg.MIN_COHORT_SIZE)
        e.loc[m, "cohort_level"] = lvl; e.loc[m, "cohort_size"] = size[m]
    m = e["cohort_level"].isna()
    e.loc[m, "cohort_level"] = "L4"; e.loc[m, "cohort_size"] = len(e)
    keys_of = dict(COHORT_LEVELS)
    e["cohort_key"] = [ " | ".join(f"{k}={r[k]}" for k in keys_of[l]) or "SEMUA"
                        for l, (_, r) in zip(e["cohort_level"], e.iterrows()) ]
    return e

EMP_C = assign_cohorts(EMP)
print("Distribusi level cohort:")
print(EMP_C["cohort_level"].value_counts().sort_index().to_string())
print(f"\nJumlah cohort unik: {EMP_C.cohort_key.nunique()}, ukuran median: {EMP_C.cohort_size.median():.0f}")
EMP_C.groupby("cohort_key")["cohort_size"].first().sort_values().head(10)

## 7. Statistik cohort per periode + posisi relatif
Statistik cohort (median, Q1, Q3 log-upah) dihitung dari **semua** employer sejenis, termasuk yang tidak lolos pre-filter UMR, supaya benchmark-nya tidak bias ke bawah.

In [ ]:
def cohort_positions(panel, emp_c, cfg=CFG):
    p = panel.merge(emp_c[["employer_id", "cohort_level", "cohort_size", "cohort_key"]], on="employer_id")
    parts = []
    for lvl, keys in COHORT_LEVELS:
        sub = p[p["cohort_level"] == lvl]
        if sub.empty:
            continue
        gk = keys + ["period"]
        st = (p.groupby(gk)["log_wage"]
                .agg(c_median="median", c_q1=lambda s: s.quantile(.25), c_q3=lambda s: s.quantile(.75), c_n="count")
                .reset_index())
        parts.append(sub.merge(st, on=gk, how="left"))
    d = pd.concat(parts, ignore_index=True)
    d["c_iqr"] = np.maximum(d["c_q3"] - d["c_q1"], cfg.IQR_FLOOR)
    valid = d["c_n"] >= cfg.MIN_COHORT_SIZE
    d["z"] = np.where(valid, (d["log_wage"] - d["c_median"]) / d["c_iqr"], np.nan)
    d["pct_vs_median"] = np.exp(d["log_wage"] - d["c_median"]) - 1
    d["cohort_median_wage"] = np.exp(d["c_median"])
    return d.sort_values(["employer_id", "period"]).reset_index(drop=True)

DETAIL = cohort_positions(PANEL, EMP_C)
DETAIL[["employer_id", "period", "avg_wage", "cohort_key", "cohort_median_wage", "z", "pct_vs_median"]].head()

## 8. Skor B per employer (UMR pre-filter → peer-group test → directional flag)
1. **INSUFFICIENT_DATA**: data upah < `MIN_PERIODS` bulan (netral).
2. **NOT_CANDIDATE**: upah median > `UMR_PREFILTER_MULT` × UMR, jadi tidak masuk kandidat awal.
3. Kandidat → **FLAGGED** kalau posisi median < −`Z_THRESHOLD` **dan** konsisten di ≥ `MIN_PERSIST` bulan.
4. Upah di atas cohort **tidak pernah** di-flag (directional).

In [ ]:
FLAGGED, NORMAL, NOT_CAND, INSUFFICIENT = "FLAGGED", "NORMAL", "NOT_CANDIDATE", "INSUFFICIENT_DATA"
rp = lambda x: "Rp" + f"{x:,.0f}".replace(",", ".") if pd.notna(x) else "-"

def make_reason(r, cfg=CFG):
    if r.status == INSUFFICIENT:
        return f"Data upah baru {int(r.n_periods)} bulan (< {cfg.MIN_PERIODS}); belum bisa dinilai."
    cohort = f"{r.cohort_key}{LEVEL_NOTE[r.cohort_level]}, n={int(r.cohort_size)}"
    if r.status == NOT_CAND:
        return (f"Upah rata-rata {rp(r.wage_median)}/orang di atas {cfg.UMR_PREFILTER_MULT}× UMR "
                f"({rp(r.umr)}); tidak masuk kandidat awal.")
    if r.status == FLAGGED:
        umr_note = (" Upah juga di bawah UMR, tetapi itu sendiri bukan dasar flag (urusan Disnaker)."
                    if pd.notna(r.umr) and r.wage_median < r.umr else "")
        return (f"Perlu dicek — indikasi lapor gaji lebih rendah dari aslinya: upah dilaporkan {rp(r.wage_median)}/orang, "
                f"{abs(r.pct_vs_median):.0%} di bawah median perusahaan sejenis ({rp(r.cohort_median_wage)}; {cohort}), "
                f"{abs(r.z_median):.1f} IQR di bawah median, konsisten di {int(r.n_below)}/{int(r.n_valid)} bulan.{umr_note}")
    direction = "di bawah" if r.pct_vs_median < 0 else "di atas"
    return (f"Upah sejalan dengan perusahaan sejenis ({abs(r.pct_vs_median):.0%} {direction} median; {cohort}).")

def score_employers(detail, cfg=CFG, use_umr=None):
    use_umr = HAS_UMR if use_umr is None else use_umr
    d = detail.copy()
    d["below"] = (d["z"] < -cfg.Z_THRESHOLD).where(d["z"].notna())
    g = d.groupby("employer_id")
    e = g.agg(n_periods=("period", "nunique"), n_valid=("z", "count"), n_below=("below", "sum"),
              z_median=("z", "median"), pct_vs_median=("pct_vs_median", "median"),
              wage_median=("avg_wage", "median"), cohort_median_wage=("cohort_median_wage", "median"),
              umr=("umr", "median"), cohort_key=("cohort_key", "first"),
              cohort_level=("cohort_level", "first"), cohort_size=("cohort_size", "first")).reset_index()
    e["frac_below"] = e["n_below"] / e["n_valid"].replace(0, np.nan)

    if use_umr and cfg.UMR_PREFILTER_MULT is not None:
        e["candidate"] = e["umr"].isna() | (e["wage_median"] <= cfg.UMR_PREFILTER_MULT * e["umr"])
    else:
        e["candidate"] = True

    enough = e["n_periods"] >= cfg.MIN_PERIODS
    flag = enough & e["candidate"] & (e["z_median"] < -cfg.Z_THRESHOLD) & (e["frac_below"] >= cfg.MIN_PERSIST)
    e["status"] = np.select([~enough, ~e["candidate"], flag], [INSUFFICIENT, NOT_CAND, FLAGGED], NORMAL)

    raw = (-e["z_median"]).clip(lower=0).fillna(0)            # directional: hanya sisi bawah
    e["score_b"] = np.where(e["status"].isin([FLAGGED, NORMAL]), raw, 0.0)
    e.loc[e["status"] == INSUFFICIENT, "score_b"] = np.nan
    e["score_b_norm"] = np.minimum(e["score_b"] / (cfg.Z_THRESHOLD * cfg.SCORE_CAP_MULT), 1.0)
    e["reason"] = [make_reason(r, cfg) for r in e.itertuples()]

    order = {FLAGGED: 0, NORMAL: 1, NOT_CAND: 2, INSUFFICIENT: 3}
    e = (e.assign(_o=e["status"].map(order))
           .sort_values(["_o", "score_b"], ascending=[True, False], na_position="last")
           .drop(columns="_o").reset_index(drop=True))
    e.insert(0, "rank", range(1, len(e) + 1))
    for c in ["z_median", "pct_vs_median", "frac_below", "score_b", "score_b_norm"]:
        e[c] = e[c].round(3)
    return e

def run_module_b(cfg=CFG):
    emp_c = assign_cohorts(EMP, cfg)
    det = cohort_positions(PANEL, emp_c, cfg)
    return score_employers(det, cfg), det

RESULT_B, DETAIL = run_module_b(CFG)
print(RESULT_B["status"].value_counts().to_string())
RESULT_B.head(15)[["rank", "employer_id", "status", "score_b", "z_median", "pct_vs_median", "reason"]]

## 9. Evaluasi vs `ground_truth.csv`
- **Recall**: dari kasus under-reporting upah yang di-inject, berapa persen ter-flag?
- **False positive rate**: dari employer bersih, berapa persen salah ter-flag?
- `NOT_CANDIDATE` dihitung sebagai *tidak di-flag*, jadi recall ikut menanggung biaya pre-filter UMR. `positif_hilang_di_prefilter` menunjukkan berapa kasus yang terbuang di tahap itu.

⚠ **Cek `POSITIVE_LABELS`**: harus hanya menangkap label under-reporting upah, bukan PDUK/sembunyiin karyawan atau tidak setor.

In [ ]:
POSITIVE_LABELS = ["WAGE", "UPAH", "GAJI", "SALARY", "PDSU", "PDUU", "UNDERPAY"]
NEGATIVE_EXCLUDE = ["HEADCOUNT", "EMPLOYEE", "KARYAWAN", "PDUK", "REMIT", "SETOR", "NONE", "CLEAN"]

def build_truth():
    if GT is None:
        return None
    col = GT[COLS["gt_label"]].astype(str)
    print("Nilai label di ground truth:", col.value_counts().to_dict())
    up = col.str.upper()
    pos = up.str.contains("|".join(POSITIVE_LABELS), na=False) & ~up.str.fullmatch("|".join(NEGATIVE_EXCLUDE))
    t = pd.DataFrame({"employer_id": GT[COLS["gt_employer"]].astype(str), "actual": pos})
    return t.groupby("employer_id", as_index=False)["actual"].any()

def evaluate(result, truth, verbose=True):
    m = result.merge(truth, on="employer_id", how="left")
    m["actual"] = m["actual"].fillna(False).astype(bool)
    ev = m[m["status"] != INSUFFICIENT]
    pred = ev["status"] == FLAGGED
    tp, fp = int((pred & ev.actual).sum()), int((pred & ~ev.actual).sum())
    fn, tn = int((~pred & ev.actual).sum()), int((~pred & ~ev.actual).sum())
    res = dict(tp=tp, fp=fp, fn=fn, tn=tn,
               precision=round(tp / (tp + fp), 3) if tp + fp else np.nan,
               recall=round(tp / (tp + fn), 3) if tp + fn else np.nan,
               false_positive_rate=round(fp / (fp + tn), 3) if fp + tn else np.nan,
               positif_hilang_di_prefilter=int((ev.actual & (ev.status == NOT_CAND)).sum()),
               positif_di_insufficient=int(m.loc[m.status == INSUFFICIENT, "actual"].sum()))
    if verbose:
        for k, v in res.items():
            print(f"  {k:28s}: {v}")
        cols = ["employer_id", "status", "z_median", "pct_vs_median", "frac_below", "reason"]
        missed, fa = ev[~pred & ev.actual], ev[pred & ~ev.actual]
        if len(missed):
            print(f"\nKasus under-reporting yang TERLEWAT ({len(missed)}):")
            print(missed[cols].head(15).to_string(index=False))
        if len(fa):
            print(f"\nEmployer bersih yang SALAH ke-flag ({len(fa)}):")
            print(fa[cols].head(15).to_string(index=False))
    return res

TRUTH = build_truth()
if TRUTH is not None:
    print(f"Positif (kasus Module B) di ground truth: {int(TRUTH.actual.sum())} employer\n")
    EVAL_B = evaluate(RESULT_B, TRUTH)
else:
    print("ground_truth.csv tidak ada — evaluasi dilewati")

## 10. Tuning threshold (sweep)
Trade-off utama:
- `Z_THRESHOLD` lebih kecil → recall naik, false positive naik.
- `UMR_PREFILTER_MULT` lebih kecil → kandidat lebih sedikit (lebih efisien), tapi under-reporter yang gaji aslinya tinggi bisa lolos. `None` = pre-filter mati (batas atas recall).

Pilih kombinasi, tulis di `ConfigB` (sel 4), lalu Run all lagi.

In [ ]:
def threshold_sweep(z_grid=(0.75, 1.0, 1.5, 2.0, 2.5), umr_grid=(1.0, 1.25, 1.5, 2.0, None)):
    emp_c = assign_cohorts(EMP, CFG)
    det = cohort_positions(PANEL, emp_c, CFG)        # cohort tidak bergantung threshold
    rows = []
    for z in z_grid:
        for u in umr_grid:
            cfg = replace(CFG, Z_THRESHOLD=z, UMR_PREFILTER_MULT=u)
            e = evaluate(score_employers(det, cfg), TRUTH, verbose=False)
            rows.append(dict(Z_THRESHOLD=z, UMR_PREFILTER_MULT=u if u is not None else "off",
                             **{k: e[k] for k in ("tp", "fp", "fn", "precision", "recall",
                                                  "false_positive_rate", "positif_hilang_di_prefilter")}))
    return pd.DataFrame(rows)

if TRUTH is not None:
    SWEEP = threshold_sweep()
    display(SWEEP) if "display" in globals() else print(SWEEP.to_string(index=False))

## 11. Visual sanity check
Garis biru = upah employer, area abu = Q1–Q3 cohort, garis putus = median cohort, garis merah = UMR.

In [ ]:
def plot_employers(ids, detail=DETAIL):
    ids = list(ids)
    if not ids:
        print("Tidak ada employer untuk di-plot"); return
    fig, axes = plt.subplots(len(ids), 1, figsize=(10, 2.5 * len(ids)), squeeze=False)
    for ax, eid in zip(axes[:, 0], ids):
        d = detail[detail.employer_id == eid]
        x = d["period"].astype(str)
        ax.fill_between(x, np.exp(d.c_q1), np.exp(d.c_q3), color="lightgray", label="cohort Q1–Q3")
        ax.plot(x, np.exp(d.c_median), "k--", lw=1, label="median cohort")
        ax.plot(x, d.avg_wage, marker="o", color="steelblue", label="upah employer")
        if d.umr.notna().any():
            ax.plot(x, d.umr, color="red", lw=1, label="UMR")
        ax.set_title(f"{eid} — {d.cohort_key.iloc[0]}", fontsize=9, loc="left")
        ax.tick_params(axis="x", rotation=45, labelsize=7); ax.legend(fontsize=7, loc="upper right")
    plt.tight_layout(); plt.show()

plot_employers(RESULT_B.loc[RESULT_B.status == FLAGGED, "employer_id"].head(4))

## 12. Simpan output ke Google Drive (`wave 0.B`)
`module_b_scores.csv` → nanti digabung dengan Module A & C (pakai kolom `score_b_norm`, 0–1).

In [ ]:
OUT_DIR = Path(os.environ.get("HK_OUT_DIR", OUT_DRIVE if IN_COLAB else "output_module_b"))
if IN_COLAB and str(OUT_DIR).startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESULT_B.to_csv(OUT_DIR / "module_b_scores.csv", index=False)
DETAIL.assign(period=DETAIL["period"].astype(str)) \
      .drop(columns=["total", "umr_p", "umr_m"], errors="ignore") \
      .to_csv(OUT_DIR / "module_b_period_detail.csv", index=False)
EMP_C.to_csv(OUT_DIR / "module_b_cohorts.csv", index=False)
if TRUTH is not None:
    SWEEP.to_csv(OUT_DIR / "module_b_threshold_sweep.csv", index=False)
with open(OUT_DIR / "module_b_config.txt", "w") as f:
    f.write(str(asdict(CFG)) + "\n")

print("Tersimpan di:", OUT_DIR)
for f in sorted(OUT_DIR.glob("*")):
    print("  ✓", f.name)